In [ ]:
!pip install pyspark


###Question1

In [ ]:


from pyspark.sql.functions import col
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('Team_combination').getOrCreate()
data=[('RCB',1),
      ('CSK',2),
       ('MI',3),
       ('PBKS',4)]
df=spark.createDataFrame(data,['col1','id'])
df.show()

+----+---+
|col1| id|
+----+---+
| RCB|  1|
| CSK|  2|
|  MI|  3|
|PBKS|  4|
+----+---+



In [ ]:
team1.show()

+----+---+
|col1| id|
+----+---+
| RCB|  1|
| CSK|  2|
|  MI|  3|
|PBKS|  4|
+----+---+



In [ ]:
team2.show()

+----+---+
|col1| id|
+----+---+
| RCB|  1|
| CSK|  2|
|  MI|  3|
|PBKS|  4|
+----+---+



In [ ]:
team1=df.alias("team1")
team2=df.alias("team2")
result=team1.crossJoin(team2)\
.filter(col("team1.id")<col("team2.id"))\
.select(col("team1.col1").alias("team1"),col("team2.col1").alias("team2"))
result.show()

+-----+-----+
|team1|team2|
+-----+-----+
|  RCB|  CSK|
|  RCB|   MI|
|  RCB| PBKS|
|  CSK|   MI|
|  CSK| PBKS|
|   MI| PBKS|
+-----+-----+



###Question2

In [ ]:

from pyspark.sql.window import Window
from pyspark.sql.functions import col,lag
from pyspark.sql import SparkSession
spark=SparkSession.builder.getOrCreate()
data=[("A",),("B",),("B",),("A",),("A",),("A",),("B",),("C",),("D",),("C",),("C",),("C",),("C",)]

df=spark.createDataFrame(data,['element'])

In [ ]:
df.show()

+-------+
|element|
+-------+
|      A|
|      B|
|      B|
|      A|
|      A|
|      A|
|      B|
|      C|
|      D|
|      C|
|      C|
|      C|
|      C|
+-------+



In [ ]:
df1=df.withColumn("prev1",lag("element",1).over(Window))
df2=df.withColumn("prev2",lag("element",2).over(Window))
df1.show()
df2.show()


PySparkTypeError: [NOT_WINDOWSPEC] Argument `window` should be a WindowSpec, got type.

###Question3

In [ ]:

from pyspark.sql.functions import col,when
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("price_category").getOrCreate()

data=[
    ("laptop",800),
    ("Mouse",25),
    ("Keyboard",150),
    ("Monitor",300),
]

df=spark.createDataFrame(data,["product","price"])
df.show()

+--------+-----+
| product|price|
+--------+-----+
|  laptop|  800|
|   Mouse|   25|
|Keyboard|  150|
| Monitor|  300|
+--------+-----+



In [ ]:
result=df.withColumn("price_category",
                     when(col("price")<100,"low")
                  .when((col("price")>=100)&(col("price")<500),"Medium")
                  .otherwise("High"))
result.show()

+--------+-----+--------------+
| product|price|price_category|
+--------+-----+--------------+
|  laptop|  800|          High|
|   Mouse|   25|           low|
|Keyboard|  150|        Medium|
| Monitor|  300|        Medium|
+--------+-----+--------------+



###Question4

In [ ]:
from pyspark.sql.functions import col,when
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("EmployeeManager").getOrCreate()
data = [
    (1, "Alice", 3, "5/10/2018"),
    (2, "Bob", 3, "3/15/2019"),
    (3, "Charlie", 4, "1/1/2020"),
    (4, "David", None, "8/20/2017"),
    (5, "Eva", 3, "11/5/2017")
]

columns=["emp_id", "emp_name", "manager_id", "join_date"]
df = spark.createDataFrame(data, columns)
df.show()

+------+--------+----------+---------+
|emp_id|emp_name|manager_id|join_date|
+------+--------+----------+---------+
|     1|   Alice|         3|5/10/2018|
|     2|     Bob|         3|3/15/2019|
|     3| Charlie|         4| 1/1/2020|
|     4|   David|      NULL|8/20/2017|
|     5|     Eva|         3|11/5/2017|
+------+--------+----------+---------+



In [ ]:
# Convert string to date
from pyspark.sql.functions import to_date
df = df.withColumn(
    "join_date",
    to_date(col("join_date"), "M/d/yyyy")
)

# Create two aliases
employee = df.alias("employee")
manager = df.alias("manager")
#self join
result = employee.join(
    manager,
    col("employee.manager_id") == col("manager.emp_id"),
    "inner"
)

# Employee joined before manager
result = result.filter(
    (col("employee.join_date") < col("manager.join_date"))&
    (col("employee.emp_name") !="Bob")
)

# Select required columns
result = result.select(
    col("employee.emp_name").alias("emp_name"),
    col("manager.emp_name").alias("manager_name")
)
result.show()

+--------+------------+
|emp_name|manager_name|
+--------+------------+
|   Alice|     Charlie|
|     Eva|     Charlie|
+--------+------------+

